# Download Required GAMA Data

Run this notebook once to re-fetch all FITS files needed by the analysis notebooks.

**Requirements:**
```
pip install astroquery astropy
```

Files are saved to `../../GAMA_DATA/` (i.e. `MSc_Project/GAMA_DATA/`).

GAMA data is public — no login required for the tables used here.
See: https://www.gama-survey.org/dr4/


In [ ]:
from astroquery.gama import GAMA
from astropy.table import Table
import os

DATA_DIR = '../../GAMA_DATA'
os.makedirs(DATA_DIR, exist_ok=True)
print(f'Saving to: {os.path.abspath(DATA_DIR)}')

In [ ]:
def fetch_and_save(sql, filename, description=''):
    path = os.path.join(DATA_DIR, filename)
    if os.path.exists(path):
        print(f'Already exists, skipping: {filename}')
        return
    print(f'Fetching {description or filename} ...')
    result = GAMA.query_sql(sql)
    t = Table(result)
    t.write(path, format='fits', overwrite=True)
    print(f'  Saved {len(t)} rows → {filename}')

print('fetch_and_save defined')

## Core tables

In [ ]:
# Stellar masses, colours, metallicity
fetch_and_save(
    'SELECT CATAID, uminusr, deluminusr, logmstar, dellogmstar, metal, delmetal, \
            logLWage, logage, Z, nQ \
     FROM StellarMassesv19',
    'StellarMassesv19.fits',
    'StellarMassesv19'
)

In [ ]:
# Environment measures
fetch_and_save(
    'SELECT CATAID, CountInCyl, CountInCylFlag, DistanceTo5nn, DistanceTo5nnFlag, \
            SurfaceDensity, SurfaceDensityFlag, AGEDenPar, AGEDenParFlag \
     FROM EnvironmentMeasuresv05',
    'EnvironmentMeasuresv05.fits',
    'EnvironmentMeasuresv05'
)

In [ ]:
# Group catalogue — halo masses, TotRmag, group properties
fetch_and_save(
    'SELECT GroupID, Zfof, Nfof, TotRmag, MassA, MassAfunc, \
            Rad50, Rad100, VelDisp, RelDen, IterCenRA, IterCenDec, IterCenZ, \
            BCGCATAID, Rgap, RadKurt, AxRat, VelSkew, VelKurt \
     FROM G3CFoFGroupv10',
    'G3CFoFGroupv10.fits',
    'G3CFoFGroupv10 (group catalogue)'
)

In [ ]:
# Galaxy group membership
fetch_and_save(
    'SELECT CATAID, GroupID, RankBCG, SepBCG, CoSepBCG, \
            RankCen, SepCen, CoSepCen, \
            RankIterCen, SepIterCen, CoSepIterCen, \
            Rpetro, Z \
     FROM G3CGalv10',
    'G3CGalv10.fits',
    'G3CGalv10 (galaxy group membership)'
)

In [ ]:
# Visual morphology
fetch_and_save(
    'SELECT CATAID, ELLIPTICAL, ELLIPTICAL_CODE, HUBBLE_TYPE_CODE, \
            P_EL, P_CS, P_EL_DEBIASED, P_CS_DEBIASED \
     FROM VisualMorphologyv03',
    'VisualMorphologyv03.fits',
    'VisualMorphologyv03'
)

## Verify

In [ ]:
required = [
    'StellarMassesv19.fits',
    'EnvironmentMeasuresv05.fits',
    'G3CFoFGroupv10.fits',
    'G3CGalv10.fits',
    'VisualMorphologyv03.fits',
]
print('File check:')
all_ok = True
for f in required:
    path = os.path.join(DATA_DIR, f)
    exists = os.path.exists(path)
    if exists:
        t = Table.read(path)
        print(f'  OK  {f:45s} ({len(t):,} rows)')
    else:
        print(f'  MISSING  {f}')
        all_ok = False

print()
print('All files ready — proceed to PartialCorrelation_HaloMassControl.ipynb' if all_ok else 'Some files missing — re-run cells above.')